# Fine-tuning Qwen3.5-4B di Google Colab

Notebook ini mandiri: tidak perlu menghubungkan Colab ke GitHub. Panduan lengkapnya ada di [PANDUAN_TRAINING.md](https://github.com/PvzKing/Dataset/blob/main/docs/PANDUAN_TRAINING.md).

1. Pilih GPU di menu *Runtime → Change runtime type*. T4 tersedia gratis.
2. Jalankan sel **1–4** berurutan. Konfigurasi ada di blok `CONFIG` di sel 4.
3. Pilih salah satu cara:
   - **Mode A:** jalankan sendiri dengan satu sel.
   - **Mode B:** biarkan **Claude** yang menjalankan, memantau, dan memperbaiki masalah lewat terminal.

Data `own` diunduh otomatis dari repo publik. Bisa juga upload `train.jsonl` ke panel *Files* (`/content`).
Semua hasil disimpan ke Google Drive. Jika sesi putus, jalankan ulang sel 1–4 lalu mode yang sama, dan training
**dilanjutkan dari checkpoint terakhir**.

## 1. Cek GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Instalasi
Butuh ±2–4 menit. Jika Colab meminta *Restart session*, klik tombolnya lalu lanjut ke sel 3. Sel ini tidak perlu diulang.

In [ ]:
%%capture
!pip install --upgrade unsloth

## 3. Mount Google Drive
Pilih akun Google Anda, lalu izinkan akses. Hasil training disimpan di `MyDrive/finetune-id`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Script training
Sel ini **hanya menyimpan** script ke `/content/train_unsloth.py`; training belum berjalan. Ubah blok `CONFIG` jika
perlu, lalu jalankan sel ini. Jika `CONFIG` diubah lagi nanti, jalankan ulang sel ini.

In [ ]:
%%writefile /content/train_unsloth.py
"""Fine-tuning LoRA Qwen3.5-4B dengan Unsloth dalam satu file: data, training, uji coba, dan ekspor GGUF.

Script ini mandiri: tidak butuh clone repo. Jika train.jsonl tidak ditemukan, file diunduh dari repo GitHub publik.
Cara memakai di Google Colab (pilih GPU di Runtime → Change runtime type):

    Sel 1:  !pip install --upgrade unsloth
    Sel 2:  tempel seluruh isi file ini, ubah CONFIG jika perlu, lalu jalankan. Drive di-mount otomatis, lalu
            training, penyimpanan adapter, dan pembuatan website uji berjalan berurutan. Untuk menjalankan tahap lain
            (misalnya uji ulang atau ekspor GGUF), ganti CONFIG["command"] lalu jalankan sel itu lagi.

Atau simpan sebagai file (sel diawali `%%writefile /content/train_unsloth.py`) lalu jalankan dari terminal/sel:

    python train_unsloth.py info       # tampilkan konfigurasi dan lokasi hasil
    python train_unsloth.py prepare    # cek data tanpa GPU
    python train_unsloth.py train      # training + simpan adapter + website uji (bawaan, lihat CONFIG["command"])
    python train_unsloth.py test       # uji ulang adapter yang sudah tersimpan
    python train_unsloth.py gguf       # ekspor GGUF q4_k_m untuk Ollama / llama.cpp
    Opsi CONFIG bisa diganti lewat argumen, misalnya: --dataset mix --load-in-4bit --max-seq-len 6144

Semua hasil ditulis ke <out>/<model>-<dataset>/:
    status.json      tahap yang sedang berjalan, step, loss, ETA, atau pesan error (dibaca untuk memantau)
    checkpoints/     checkpoint training; dilanjutkan dari yang terakhir jika data dan pengaturan masih sama
    arsip/           hasil training lama yang dipindahkan karena data atau pengaturan berubah
    lora-adapter/    adapter LoRA hasil training
    contoh-*.html    website yang dibuat model dari prompt uji (contoh-*.md berisi jawaban lengkapnya)
    gguf/            hasil ekspor GGUF (opsional)

Training lanjutan (--init-adapter <folder lora-adapter>) memulai dari bobot adapter yang sudah ada, dengan optimizer
dan jadwal learning rate baru. Hasilnya ditulis ke folder terpisah berakhiran -lanjut, jadi adapter asalnya tidak
berubah. Untuk test/gguf hasil lanjutan, sertakan --init-adapter yang sama supaya folder -lanjut yang dipakai.
"""
import argparse
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from pathlib import Path

# Kurangi fragmentasi memori GPU; harus diset sebelum torch di-import.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ======================================================== CONFIG: ubah di sini ========================================
CONFIG = {
    "command": "train",             # dipakai jika tidak ada argumen: "train", "prepare", "test", "gguf", atau "info"
    "model": "qwen3.5-4b",          # "qwen3.5-4b" atau "qwen2.5-coder-7b" (pembanding)
    "dataset": "own",               # "own" = data/train.jsonl (disarankan), "mix" = ditambah dataset umum,
                                    # "bench" = ditambah kelompok bench-support (HumanEval) di data/bench-support/
    "out": "/content/drive/MyDrive/finetune-id",  # folder hasil; di Google Drive supaya aman saat sesi putus
    "data": None,                   # path train.jsonl; None = cari otomatis lalu unduh dari GitHub jika tidak ada
    "init_adapter": None,           # folder lora-adapter untuk training lanjutan; None = mulai dari model dasar
    "mix_args": "",                 # argumen tambahan mix_general.py, misalnya "--code 0 --chat 0" (hanya Aya)
    "load_in_4bit": None,           # None = otomatis: 4-bit di GPU kecil tanpa bf16 (T4), selain itu bawaan model.
                                    # True = QLoRA 4-bit (hemat ±6 GB VRAM, akurasi Qwen3.5 sedikit turun)
    "max_seq_len": None,            # None = otomatis: 6144 di T4, 8192 di GPU lain. Sampel lebih panjang DIBUANG
    "epochs": None,                 # None = 3 untuk own, 2 untuk mix
    "lr": 1e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "batch_size": 1,
    "grad_accum": 8,                # batch efektif = batch_size × grad_accum
    "save_steps": 10,               # checkpoint setiap N step
    "max_new_tokens": 6000,         # batas panjang jawaban saat uji
    "no_test": False,               # True = lewati pembuatan website uji setelah training
    "seed": 42,
}
# ======================================================================================================================

REPO_RAW = "https://raw.githubusercontent.com/PvzKing/Dataset/main"
EOS = "<|im_end|>"
IN_NOTEBOOK = "ipykernel" in sys.modules  # True jika file ini ditempel langsung ke sel Colab/Jupyter
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

MODELS = {
    # Salinan Unsloth dari Qwen/Qwen3.5-4B. LoRA 16-bit, karena Unsloth tidak menyarankan QLoRA 4-bit untuk Qwen3.5.
    # Di GPU tanpa bf16 (T4), Unsloth otomatis melatih Qwen3.5 dalam float32 karena fp16 menghasilkan NaN.
    "qwen3.5-4b": {
        "name": "unsloth/Qwen3.5-4B",
        "tokenizer": "Qwen/Qwen3.5-4B",
        "load_in_4bit": False,
        "template_kwargs": {"enable_thinking": False},
    },
    # Pembanding: Qwen/Qwen2.5-Coder-7B-Instruct yang sudah dikuantisasi 4-bit (QLoRA), supaya muat di T4.
    "qwen2.5-coder-7b": {
        "name": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        "tokenizer": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "load_in_4bit": True,
        "template_kwargs": {},
    },
}

TEST_PROMPTS = {
    "bengkel": "Buatkan landing page satu file HTML untuk bengkel motor \"Gaspol Motor\" di Yogyakarta: hero dengan "
               "tombol booking servis, daftar layanan dan harga, testimoni, jam buka, dan form booking dengan validasi. "
               "Responsif dan tanpa library eksternal.",
    "absensi": "Buat aplikasi absensi kelas dalam satu file HTML: tambah siswa, tandai hadir/izin/sakit/alfa per tanggal, "
               "rekap persentase kehadiran, simpan di localStorage, dan tombol ekspor CSV.",
}


# ---------------------------------------------------------------- status

class Status:
    """Tulis kondisi terkini ke status.json supaya proses yang berjalan di latar belakang mudah dipantau."""

    def __init__(self, path):
        self.path = Path(path)
        self.data = {}

    def update(self, **fields):
        self.data.update(fields, updated=time.strftime("%Y-%m-%d %H:%M:%S"))
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps(self.data, ensure_ascii=False, indent=1), encoding="utf-8")
        tmp.replace(self.path)
        print(f"[status] {self.data.get('stage')}: {self.data.get('message', '')}", flush=True)


# ---------------------------------------------------------------- data

def ensure_drive(out):
    """Pastikan Google Drive ter-mount jika folder hasil ada di Drive."""
    if not str(out).startswith("/content/drive") or os.path.isdir("/content/drive/MyDrive"):
        return
    if IN_NOTEBOOK:
        from google.colab import drive
        drive.mount("/content/drive")
    else:
        raise RuntimeError("Google Drive belum ter-mount: jalankan `from google.colab import drive; "
                           "drive.mount('/content/drive')` di sel notebook, lalu ulangi")


def download(url, dest):
    print(f"mengunduh {url}", flush=True)
    dest = Path(dest)
    tmp = dest.with_suffix(dest.suffix + ".part")
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(dest)
    return dest


def find_own_data(args):
    """Cari train.jsonl: --data, folder script/repo, folder kerja; jika tidak ada, unduh versi terbaru dari GitHub."""
    candidates = [args.data] if args.data else []
    candidates += [HERE / "train.jsonl", HERE.parent / "data/train.jsonl", Path.cwd() / "train.jsonl",
                   Path.cwd() / "data/train.jsonl"]
    for path in candidates:
        if path and Path(path).is_file():
            return Path(path)
    if args.data:
        raise FileNotFoundError(f"file data tidak ditemukan: {args.data}")
    # Selalu unduh ulang supaya perubahan dataset di GitHub ikut terpakai; salinan lama hanya dipakai jika offline.
    cached = Path(args.out) / "train.jsonl"
    try:
        return download(f"{REPO_RAW}/data/train.jsonl", cached)
    except OSError as e:
        if not cached.is_file():
            raise
        print(f"unduhan gagal ({e}), memakai salinan lama {cached}", flush=True)
        return cached


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


BENCH_FILES = ["humaneval.jsonl"]  # isi kelompok bench-support di data/bench-support/


def bench_files(args):
    """File bench-support lokal (repo) atau unduhan terbaru dari GitHub, disimpan di folder hasil."""
    files = []
    for name in BENCH_FILES:
        local = HERE.parent / "data" / "bench-support" / name
        if local.is_file():
            files.append(local)
            continue
        cached = Path(args.out) / "bench-support" / name
        cached.parent.mkdir(parents=True, exist_ok=True)
        try:
            files.append(download(f"{REPO_RAW}/data/bench-support/{name}", cached))
        except OSError as e:
            if not cached.is_file():
                raise
            print(f"unduhan gagal ({e}), memakai salinan lama {cached}", flush=True)
            files.append(cached)
    return files


def data_file_for(args):
    own = find_own_data(args)
    if args.dataset == "own":
        return own
    if args.dataset == "bench":
        # own + bench-support digabung ke satu file di folder hasil; ditulis ulang hanya jika isinya berubah,
        # supaya fingerprint dan resume checkpoint tetap konsisten.
        parts = [own, *bench_files(args)]
        merged = "".join(Path(p).read_text(encoding="utf-8").rstrip("\n") + "\n" for p in parts)
        out = Path(args.out) / "train_bench.jsonl"
        if not out.is_file() or out.read_text(encoding="utf-8") != merged:
            out.write_text(merged, encoding="utf-8")
        print(f"data bench: {' + '.join(Path(p).name for p in parts)}", flush=True)
        return out
    # Data campuran disimpan di folder hasil supaya sesi berikutnya memakai file yang sama persis (penting untuk resume),
    # dan dibuat ulang otomatis jika data own, mix_args, atau seed berubah.
    mix = Path(args.out) / "train_mix.jsonl"
    source = Path(args.out) / "train_mix.source"
    key = f"{sha256_file(own)} {args.mix_args} {args.seed}"
    if not mix.exists() or not source.exists() or source.read_text().strip() != key:
        mixer = HERE / "mix_general.py"
        if not mixer.exists():
            mixer = download(f"{REPO_RAW}/scripts/mix_general.py", Path(args.out) / "mix_general.py")
        print("membuat data campuran ...", flush=True)
        subprocess.run([sys.executable, str(mixer), "--input", str(own), "--output", str(mix),
                        "--seed", str(args.seed), *args.mix_args.split()], check=True)
        source.write_text(key)
    return mix


def load_rows(data_file):
    """Baca JSONL dan periksa strukturnya: (system) lalu user/assistant bergantian, diakhiri assistant."""
    rows, errors, ids = [], [], set()
    for no, line in enumerate(open(data_file, encoding="utf-8"), 1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"baris {no}: JSON tidak valid ({e})")
            continue
        rid, messages = row.get("id", f"baris-{no}"), row.get("messages")
        if rid in ids:
            errors.append(f"{rid}: id duplikat")
        ids.add(rid)
        if not isinstance(messages, list) or not messages:
            errors.append(f"{rid}: 'messages' kosong atau bukan list")
            continue
        turns = messages[1:] if messages[0].get("role") == "system" else messages
        roles = [m.get("role") for m in turns]
        if not turns or roles != ["user", "assistant"] * (len(turns) // 2) or len(turns) % 2:
            errors.append(f"{rid}: urutan role harus user/assistant bergantian dan diakhiri assistant")
        if any(not isinstance(m.get("content"), str) or not m["content"].strip() for m in messages):
            errors.append(f"{rid}: ada content yang kosong")
        rows.append(row)
    if errors:
        raise ValueError(f"{len(errors)} masalah di {data_file}:\n  " + "\n  ".join(errors[:20]))
    print(f"data valid: {len(rows)} sampel dari {data_file}", flush=True)
    return rows


def unwrap_tokenizer(tokenizer):
    """Model multimodal bisa mengembalikan processor; training teks cukup memakai tokenizer di dalamnya."""
    if hasattr(tokenizer, "tokenizer"):
        processor, tokenizer = tokenizer, tokenizer.tokenizer
        if not getattr(tokenizer, "chat_template", None):
            tokenizer.chat_template = processor.chat_template
    return tokenizer


def render(tokenizer, template_kwargs, messages, **kw):
    return tokenizer.apply_chat_template(messages, tokenize=False, **template_kwargs, **kw)


def to_text(tokenizer, template_kwargs, messages):
    """Jawaban terakhir ditempel tepat setelah prompt inferensi, jadi format training sama dengan saat dipakai
    (termasuk blok <think> kosong pada mode non-thinking Qwen3.5)."""
    *history, last = messages
    return render(tokenizer, template_kwargs, history, add_generation_prompt=True) + last["content"] + EOS


def build_texts(tokenizer, template_kwargs, rows, max_seq_len):
    """Kembalikan (texts, lengths, dropped). Sampel yang lebih panjang dari max_seq_len dibuang, tidak dipotong."""
    texts, lengths, dropped = [], [], []
    for row in rows:
        text = to_text(tokenizer, template_kwargs, row["messages"])
        n = len(tokenizer(text, add_special_tokens=False)["input_ids"])
        if n > max_seq_len:
            dropped.append((row["id"], n))
            continue
        texts.append(text)
        lengths.append(n)
    if not texts:
        raise ValueError(f"semua sampel lebih panjang dari max_seq_len={max_seq_len}")
    kept = [row["id"] for row in rows if row["id"] not in {d[0] for d in dropped}]
    longest = sorted(zip(lengths, kept), reverse=True)[:3]
    print("sampel terpanjang: " + ", ".join(f"{rid} ({n:,})" for n, rid in longest), flush=True)
    return texts, lengths, dropped


def summarize(texts, lengths, dropped, args, bf16):
    total_steps = math.ceil(len(texts) / (args.batch_size * args.grad_accum)) * args.epochs
    trained_tokens = sum(lengths) * args.epochs
    lo, hi = (300, 900) if bf16 else (60, 200)  # token/detik, perkiraan kasar (T4 tanpa bf16 jauh lebih lambat)
    summary = {
        "samples": len(texts),
        "dropped": dropped,
        "tokens_per_epoch": sum(lengths),
        "longest": max(lengths),
        "epochs": args.epochs,
        "total_steps": total_steps,
        "trained_tokens": trained_tokens,
        "estimate_min": [round(trained_tokens / hi / 60), round(trained_tokens / lo / 60)],
    }
    print(f"{len(texts)} sampel dipakai, {len(dropped)} dibuang karena > {args.max_seq_len} token {dropped[:5]}")
    print(f"token per epoch: {sum(lengths):,} | terpanjang: {max(lengths):,} | rata-rata: {sum(lengths) // len(lengths):,}")
    print(f"{args.epochs:g} epoch = {total_steps} step, ±{trained_tokens:,} token dilatih")
    print(f"perkiraan waktu ({lo}–{hi} token/detik): {summary['estimate_min'][0]}–{summary['estimate_min'][1]} menit")
    print("\ncontoh awal teks training:\n" + texts[0][:400] + "\n", flush=True)
    return summary


# ---------------------------------------------------------------- model

def load_model(cfg, args, model_name=None):
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name or cfg["name"],
        max_seq_length=args.max_seq_len,
        dtype=None,
        load_in_4bit=args.load_in_4bit,
        load_in_16bit=not args.load_in_4bit,
    )
    return model, unwrap_tokenizer(tokenizer)


def generate_tests(model, tokenizer, cfg, run_dir, max_new_tokens):
    from transformers import TextStreamer
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    results = {}
    for key, prompt in TEST_PROMPTS.items():
        text = render(tokenizer, cfg["template_kwargs"], [{"role": "user", "content": prompt}], add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
        start = time.time()
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.05,
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
        (run_dir / f"contoh-{key}.md").write_text(answer, encoding="utf-8")
        has_html = "```html" in answer
        if has_html:
            page = answer.split("```html", 1)[1].split("```", 1)[0]
            (run_dir / f"contoh-{key}.html").write_text(page, encoding="utf-8")
        results[key] = {"tokens": len(new_tokens), "seconds": round(time.time() - start), "html": has_html,
                        "finished": len(new_tokens) < max_new_tokens}
        print(f"\n[{key}] {results[key]}", flush=True)
    return results


def load_init_adapter(model, args):
    """Isi LoRA yang baru dibuat dengan bobot adapter lama. Model disiapkan lewat get_peft_model seperti training biasa
    (termasuk gradient checkpointing yang dibutuhkan T4), lalu bobotnya ditimpa, dan semua bobot harus cocok."""
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file

    folder = Path(args.init_adapter)
    config_file, weights_file = folder / "adapter_config.json", folder / "adapter_model.safetensors"
    if not config_file.is_file() or not weights_file.is_file():
        raise FileNotFoundError(f"adapter tidak lengkap di {folder}: butuh adapter_config.json dan adapter_model.safetensors")
    old = json.loads(config_file.read_text())
    if (old.get("r"), old.get("lora_alpha")) != (args.lora_r, args.lora_alpha):
        raise ValueError(f"adapter lama memakai r={old.get('r')}, alpha={old.get('lora_alpha')}; samakan --lora-r dan "
                         f"--lora-alpha dengan nilai itu")
    state = load_file(str(weights_file))
    result = set_peft_model_state_dict(model, state)
    missing = [k for k in result.missing_keys if "lora_" in k]
    if missing or result.unexpected_keys:
        raise ValueError(f"bobot adapter tidak cocok dengan model: {len(missing)} bobot LoRA tidak terisi "
                         f"{missing[:3]}, {len(result.unexpected_keys)} bobot tidak dikenal {result.unexpected_keys[:3]}")
    print(f"melanjutkan dari {folder}: {len(state)} tensor LoRA dimuat", flush=True)


# ---------------------------------------------------------------- resume

# Pengaturan yang menentukan hasil training; checkpoint hanya dilanjutkan jika semuanya sama.
TRAINING_KEYS = ["model", "init_adapter", "dataset", "mix_args", "load_in_4bit", "max_seq_len", "epochs", "lr", "lora_r",
                 "lora_alpha", "batch_size", "grad_accum", "seed"]


def archive_stale_run(run_dir, ckpt_dir, fingerprint):
    """Jika checkpoint lama berasal dari data atau pengaturan lain, pindahkan hasil lama ke arsip/ supaya training
    dimulai dari awal (melanjutkannya akan diam-diam memakai bobot lama)."""
    fp_file = ckpt_dir / "fingerprint.json"
    if not ckpt_dir.exists() or (fp_file.exists() and fp_file.read_text() == fingerprint):
        return
    old = [p for p in run_dir.iterdir() if p.name in ("checkpoints", "lora-adapter", "gguf") or p.name.startswith("contoh-")]
    if not old:
        return
    dest = run_dir / "arsip" / time.strftime("%Y%m%d-%H%M%S")
    dest.mkdir(parents=True)
    for p in old:
        shutil.move(str(p), str(dest / p.name))
    print(f"data atau pengaturan berubah: hasil lama dipindah ke {dest}, training dimulai dari awal", flush=True)


# ---------------------------------------------------------------- commands

def cmd_prepare(args, cfg, status, run_dir):
    from transformers import AutoTokenizer

    rows = load_rows(data_file_for(args))
    tokenizer = unwrap_tokenizer(AutoTokenizer.from_pretrained(cfg["tokenizer"]))
    texts, lengths, dropped = build_texts(tokenizer, cfg["template_kwargs"], rows, args.max_seq_len)
    try:
        import torch
        bf16 = native_bf16(torch)
    except ImportError:
        bf16 = False
    summary = summarize(texts, lengths, dropped, args, bf16)
    status.update(stage="prepared", message="data siap, belum training", data=summary)


def cmd_train(args, cfg, status, run_dir):
    status.update(stage="loading", message=f"memuat {cfg['name']}")
    data_file = data_file_for(args)
    rows = load_rows(data_file)

    from unsloth import FastLanguageModel, is_bfloat16_supported  # harus di-import sebelum trl/transformers
    from datasets import Dataset
    from transformers import TrainerCallback
    from transformers.trainer_utils import get_last_checkpoint
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only

    model, tokenizer = load_model(cfg, args)
    model = FastLanguageModel.get_peft_model(
        model,
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=0,                  # 0 adalah jalur tercepat di Unsloth
        bias="none",
        target_modules=None,             # semua layer linear di attention (termasuk Gated DeltaNet) dan MLP
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        use_gradient_checkpointing="unsloth",
        random_state=args.seed,
    )
    if args.init_adapter:
        load_init_adapter(model, args)

    bf16 = is_bfloat16_supported()
    texts, lengths, dropped = build_texts(tokenizer, cfg["template_kwargs"], rows, args.max_seq_len)
    summary = summarize(texts, lengths, dropped, args, bf16)
    status.update(stage="prepared", message="data siap", data=summary)

    class StatusCallback(TrainerCallback):
        def on_train_begin(self, a, state, control, **kw):
            self.t0, self.step0 = time.time(), state.global_step

        def on_log(self, a, state, control, logs=None, **kw):
            done = state.global_step - self.step0
            elapsed = time.time() - self.t0
            eta = elapsed / done * (state.max_steps - state.global_step) / 60 if done else None
            status.update(stage="training", message=f"step {state.global_step}/{state.max_steps}",
                          step=state.global_step, max_steps=state.max_steps, epoch=round(state.epoch or 0, 2),
                          loss=(logs or {}).get("loss", status.data.get("loss")),
                          elapsed_min=round(elapsed / 60, 1), eta_min=round(eta, 1) if eta is not None else None)

    ckpt_dir = run_dir / "checkpoints"
    fingerprint = json.dumps({"data": sha256_file(data_file), **{k: getattr(args, k) for k in TRAINING_KEYS}},
                             sort_keys=True)
    archive_stale_run(run_dir, ckpt_dir, fingerprint)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    (ckpt_dir / "fingerprint.json").write_text(fingerprint)
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=Dataset.from_dict({"text": texts}),
        callbacks=[StatusCallback()],
        args=SFTConfig(
            output_dir=str(ckpt_dir),
            dataset_text_field="text",
            max_length=args.max_seq_len,
            packing=False,
            per_device_train_batch_size=args.batch_size,
            gradient_accumulation_steps=args.grad_accum,
            num_train_epochs=args.epochs,
            learning_rate=args.lr,
            lr_scheduler_type="cosine",
            warmup_steps=max(1, round(0.1 * summary["total_steps"])),  # 10% dari total step
            weight_decay=0.01,
            optim="adamw_8bit",
            fp16=not bf16,               # Unsloth mengganti ke float32 sendiri jika model tidak aman di fp16
            bf16=bf16,
            logging_steps=1,
            save_strategy="steps",
            save_steps=args.save_steps,
            save_total_limit=2,
            seed=args.seed,
            report_to="none",
            dataset_num_proc=2,
        ),
    )
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")

    labels = trainer.train_dataset[0]["labels"]
    trained = tokenizer.decode([t for t in labels if t != -100])
    print(f"cek masking: {sum(t != -100 for t in labels)} dari {len(labels)} token sampel pertama ikut dilatih:")
    print(trained[:300], "...\n", flush=True)

    last = get_last_checkpoint(str(ckpt_dir)) if ckpt_dir.exists() else None
    status.update(stage="training", message=f"mulai dari {last or 'awal'}", resumed_from=last)
    stats = trainer.train(resume_from_checkpoint=last)

    status.update(stage="saving", message="menyimpan adapter")
    adapter_dir = run_dir / "lora-adapter"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    result = {"train_runtime_min": round(stats.metrics["train_runtime"] / 60, 1),
              "train_loss": round(stats.metrics["train_loss"], 4), "adapter": str(adapter_dir)}
    status.update(stage="trained", message="adapter tersimpan", result=result)

    if not args.no_test:
        status.update(stage="testing", message="membuat website dari prompt uji")
        status.update(stage="done", message="selesai", tests=generate_tests(model, tokenizer, cfg, run_dir, args.max_new_tokens))
    else:
        status.update(stage="done", message="selesai (tanpa uji)")


def cmd_test(args, cfg, status, run_dir):
    status.update(stage="testing", message="memuat adapter")
    model, tokenizer = load_model(cfg, args, model_name=str(run_dir / "lora-adapter"))
    status.update(stage="done", message="uji selesai", tests=generate_tests(model, tokenizer, cfg, run_dir, args.max_new_tokens))


def cmd_gguf(args, cfg, status, run_dir):
    status.update(stage="exporting", message="menggabungkan adapter dan membuat GGUF q4_k_m")
    model, tokenizer = load_model(cfg, args, model_name=str(run_dir / "lora-adapter"))
    model.save_pretrained_gguf(str(run_dir / "gguf"), tokenizer, quantization_method="q4_k_m")
    status.update(stage="done", message="GGUF tersimpan", gguf=str(run_dir / "gguf"))


def native_bf16(torch):
    """bf16 asli (Ampere ke atas). torch.cuda.is_bf16_supported() bernilai True juga di T4 karena emulasi."""
    return torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8


def detect_small_gpu():
    """True jika GPU tidak mendukung bf16 dan VRAM < 20 GB (misalnya T4): Qwen3.5 dilatih dalam float32 di sana,
    dan di T4 16-bit maupun 4-bit dengan konteks 8192 terbukti kehabisan memori."""
    try:
        import torch
        if not torch.cuda.is_available():
            return False
        return not native_bf16(torch) and torch.cuda.get_device_properties(0).total_memory < 20e9
    except ImportError:
        return False


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("command", nargs="?", choices=["info", "prepare", "train", "test", "gguf"])
    parser.add_argument("--model", choices=sorted(MODELS))
    parser.add_argument("--dataset", choices=["own", "mix", "bench"])
    parser.add_argument("--out", help="folder hasil, misalnya di Google Drive")
    parser.add_argument("--data", help="path train.jsonl (bawaan: cari otomatis, lalu unduh dari GitHub)")
    parser.add_argument("--init-adapter", help="folder lora-adapter untuk training lanjutan (hasil ke folder -lanjut)")
    parser.add_argument("--mix-args", help='argumen tambahan mix_general.py, tulis dengan =, misalnya --mix-args="--code 0 --chat 0"')
    parser.add_argument("--load-in-4bit", action=argparse.BooleanOptionalAction, default=None,
                        help="QLoRA 4-bit (hemat VRAM); --no-load-in-4bit memaksa bawaan model")
    parser.add_argument("--max-seq-len", type=int, help="sampel yang lebih panjang dibuang")
    parser.add_argument("--epochs", type=float, help="bawaan: 3 untuk own, 2 untuk mix")
    parser.add_argument("--lr", type=float)
    parser.add_argument("--lora-r", type=int)
    parser.add_argument("--lora-alpha", type=int)
    parser.add_argument("--batch-size", type=int)
    parser.add_argument("--grad-accum", type=int)
    parser.add_argument("--save-steps", type=int)
    parser.add_argument("--max-new-tokens", type=int, help="batas panjang jawaban saat uji")
    parser.add_argument("--no-test", action="store_true", default=None, help="lewati website uji setelah training")
    parser.add_argument("--seed", type=int)
    # Saat ditempel ke sel notebook, argumen kernel Jupyter diabaikan dan CONFIG yang dipakai.
    args = parser.parse_args([] if IN_NOTEBOOK else None)
    for key, value in CONFIG.items():
        if getattr(args, key, None) is None:
            setattr(args, key, value)
    if args.command not in ("info", "prepare", "train", "test", "gguf"):
        raise ValueError(f"command tidak dikenal: {args.command!r}")
    if args.epochs is None:
        args.epochs = 2 if args.dataset == "mix" else 3
    if args.load_in_4bit is None or args.max_seq_len is None:
        small = detect_small_gpu()
        if args.load_in_4bit is None:
            args.load_in_4bit = MODELS[args.model]["load_in_4bit"] or small
        if args.max_seq_len is None:
            args.max_seq_len = 6144 if small else 8192
        if small:
            print("GPU kecil tanpa bf16 terdeteksi (misalnya T4): memakai pengaturan hemat memori "
                  f"(load_in_4bit={args.load_in_4bit}, max_seq_len={args.max_seq_len})", flush=True)
    return args


def main():
    args = parse_args()
    cfg = MODELS[args.model]
    # Hasil QLoRA 4-bit untuk model yang bawaannya 16-bit disimpan terpisah supaya checkpoint tidak tercampur.
    suffix = "-4bit" if args.load_in_4bit and not cfg["load_in_4bit"] else ""
    if args.init_adapter:
        suffix += "-lanjut"
    run_dir = Path(args.out) / f"{args.model}-{args.dataset}{suffix}"
    if args.command == "info":
        print(json.dumps({"model": args.model, "dataset": args.dataset, "load_in_4bit": args.load_in_4bit,
                          "max_seq_len": args.max_seq_len,
                          "epochs": args.epochs, "run_dir": str(run_dir), "status": str(run_dir / "status.json")},
                         indent=1))
        return

    ensure_drive(args.out)
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"folder hasil: {run_dir}", flush=True)
    status = Status(run_dir / "status.json")
    status.update(stage="starting", message=args.command, command=args.command, model=args.model,
                  dataset=args.dataset, load_in_4bit=args.load_in_4bit, max_seq_len=args.max_seq_len,
                  pid=os.getpid())
    commands = {"prepare": cmd_prepare, "train": cmd_train, "test": cmd_test, "gguf": cmd_gguf}
    try:
        commands[args.command](args, cfg, status, run_dir)
    except BaseException as e:
        tb = traceback.format_exc()
        hint = None
        if "out of memory" in tb.lower() and not args.load_in_4bit:
            hint = "VRAM tidak cukup: ulangi dengan --load-in-4bit --max-seq-len 6144 (di T4 keduanya diperlukan)"
        elif "out of memory" in tb.lower():
            # Di T4 dengan 4-bit, sampel 5.976 token terbukti muat dan 6.726 token kehabisan memori.
            smaller = 6144 if args.max_seq_len > 6144 else 6000 if args.max_seq_len > 6000 else 4096
            hint = (f"VRAM tidak cukup: ulangi dengan --max-seq-len {smaller}"
                    + (" (1–2 sampel web terpanjang akan terbuang)" if smaller == 6000 else
                       " (sebagian besar sampel web akan terbuang; GPU L4 lebih disarankan)" if smaller < 6000 else ""))
        elif re.search(r"\bnan\b", str(e), re.IGNORECASE):
            hint = "loss NaN: pasang ulang Unsloth terbaru; untuk Qwen3.5 di T4 Unsloth harus beralih ke float32"
        elif "drive belum ter-mount" in str(e).lower():
            hint = "mount Google Drive dari sel notebook"
        status.update(stage="error", message=f"{type(e).__name__}: {e}"[:500], hint=hint, traceback=tb[-3000:])
        raise


if __name__ == "__main__":
    main()

---
## Mode A: jalankan sendiri

Satu sel ini menjalankan seluruh proses:
1. cek data;
2. training;
3. simpan adapter;
4. buat dua website uji.

Progress dan ETA tampil di bawah sel. Biarkan tab browser tetap terbuka.

In [ ]:
!python /content/train_unsloth.py train

### Lihat hasil

In [ ]:
import html
import json
import os
import subprocess
from IPython.display import HTML, display

info = json.loads(subprocess.check_output(["python", "/content/train_unsloth.py", "info"]))
RUN_DIR = info["run_dir"]
status = json.load(open(f"{RUN_DIR}/status.json"))
print(json.dumps({k: status.get(k) for k in ("stage", "message", "result", "tests", "hint")}, indent=1, ensure_ascii=False))
for name in sorted(os.listdir(RUN_DIR)):
    if name.endswith(".html"):
        page = open(f"{RUN_DIR}/{name}", encoding="utf-8").read()
        display(HTML(f'<h3>{name}</h3><iframe srcdoc="{html.escape(page)}" '
                     'style="width:100%;height:600px;border:1px solid #ccc"></iframe>'))

---
## Mode B: Claude yang mengontrol lewat terminal

Claude Code berjalan di terminal VM Colab ini, dan seluruh kendali dilakukan lewat terminal tersebut. Claude bisa:
- menjalankan training di latar belakang;
- memantau log, `status.json`, dan GPU;
- menangani error umum: kehabisan memori, paket bermasalah, proses mati;
- memeriksa website hasil uji, lalu melaporkan semuanya ke Anda.

**Syarat:** akun Claude Pro, Max, Team, atau Enterprise (atau akun API Console).

### B1. Simpan aturan kerja Claude
Sel ini menyimpan aturan kerja Claude ke `/content/CLAUDE.md`. Claude Code membaca file itu otomatis saat dijalankan
di folder `/content`.

In [ ]:
%%writefile /content/CLAUDE.md
# Runbook: Claude menjalankan training di Google Colab

Dokumen ini ditujukan untuk Claude Code yang berjalan di terminal VM Google Colab. Notebook menyalinnya ke
`/content/CLAUDE.md`. Tugasmu: menjalankan dan memantau fine-tuning sampai selesai, memperbaiki masalah yang aman
diperbaiki sendiri, dan melaporkan hasilnya ke pengguna dalam bahasa Indonesia yang singkat. Pengguna berinteraksi
denganmu hanya lewat terminal ini, jadi tulis setiap laporan sebagai jawaban di terminal.

## Konteks

- **Script training:** `/content/train_unsloth.py`, ditulis oleh notebook. Script ini mandiri, tidak butuh repo.
  - Konfigurasi ada di blok `CONFIG` di awal file. Argumen seperti `--max-seq-len 6144` menimpa nilai `CONFIG` hanya untuk perintah itu.
  - Jangan mengubah file ini tanpa izin pengguna.
- **Google Drive:** ada di `/content/drive/MyDrive`. Semua yang tidak disimpan di Drive hilang saat runtime Colab di-reset.
- **Data `own`:** `train.jsonl` di `/content` dipakai lebih dulu. Jika tidak ada, versi terbaru diunduh ulang dari repo GitHub publik `PvzKing/Dataset` ke folder hasil setiap kali perintah dijalankan.
- **Lokasi hasil:** `python /content/train_unsloth.py info` menampilkan konfigurasi dan folder hasil `RUN_DIR`. Isi `RUN_DIR`:

  | File | Isi |
  |---|---|
  | `status.json` | `stage`, `message`, `step`/`max_steps`, `loss`, `eta_min`, `result`, `tests`; saat error juga `hint` dan `traceback` |
  | `checkpoints/` | checkpoint; perintah `train` melanjutkan dari checkpoint terakhir jika data dan pengaturannya sama |
  | `arsip/<waktu>/` | hasil lama yang dipindahkan otomatis karena data atau pengaturan berubah; training lalu dimulai dari awal. Ini normal, bukan error. |
  | folder `...-lanjut` | hasil training lanjutan (`--init-adapter <folder lora-adapter>`); adapter asal tidak diubah |
  | `lora-adapter/` | adapter hasil training |
  | `contoh-*.md`, `contoh-*.html` | jawaban model untuk prompt uji dan HTML yang diekstrak darinya |

  Nilai `stage` berurutan: `starting` → `loading` → `prepared` → `training` → `saving` → `trained` → `testing` → `done`.
  Jika gagal, nilainya `error`.

## Langkah

1. **Cek lingkungan**, lalu laporkan hasilnya dalam satu atau dua kalimat:
   ```bash
   python /content/train_unsloth.py info; nvidia-smi --query-gpu=name,memory.total --format=csv
   ls /content/drive/MyDrive >/dev/null && echo "drive OK"; python -c "import unsloth" && echo "unsloth OK"
   ```
   - **Drive belum ter-mount:** minta pengguna menjalankan sel *Mount Google Drive* di notebook. Mount butuh login lewat UI, jadi tidak bisa dilakukan dari terminal.
   - **Unsloth belum terpasang:** jalankan `pip install --upgrade unsloth`.
2. **Cek data tanpa GPU:**
   ```bash
   python /content/train_unsloth.py prepare
   ```
   Pastikan:
   - output berisi `data valid`;
   - jumlah sampel yang dibuang sesuai harapan (untuk `own` dengan 8192 seharusnya 0);
   - contoh teks training diakhiri prompt jawaban yang benar. Untuk Qwen3.5, prompt itu berisi blok `<think>` kosong. Untuk Qwen2.5 tidak ada blok `<think>`.

   Laporkan perkiraan step dan waktu ke pengguna.
3. **Jalankan training di latar belakang.** Jangan menjalankannya di foreground, karena prosesnya bisa berjam-jam:
   ```bash
   setsid nohup python /content/train_unsloth.py train > /content/train.log 2>&1 < /dev/null &
   ```
4. **Pantau** sampai `stage` bernilai `done` atau `error` (`RUN_DIR` dari langkah 1):
   ```bash
   cat RUN_DIR/status.json; tail -n 20 /content/train.log; nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv
   ```
   - **Saat menunggu:** pakai perintah tunggu maksimal ±9 menit per panggilan, misalnya `sleep 300`, atau alat pemantau latar belakang jika tersedia. Jarak antar pengecekan 5–10 menit sudah cukup.
   - **Di awal training:** pastikan log berisi baris `cek masking`, yaitu bagian yang dilatih diawali jawaban, bukan pertanyaan. Untuk Qwen3.5 di T4, log juga harus berisi `Switching to float32`.
   - **Loss** biasanya turun di beberapa puluh step pertama. Jika loss `nan` atau naik terus, hentikan dan ikuti tabel di bawah.
   - **Laporan ke pengguna** cukup saat ada perubahan berarti: training mulai, kira-kira setiap 25% progres, error, atau selesai. Isi laporan: step, loss, dan ETA.
5. **Setelah selesai**, baca `status.json` bagian `result` dan `tests`, lalu periksa setiap `contoh-*.html`:
   - **File lengkap:** diakhiri `</html>`, dan `finished: true` berarti jawaban tidak terpotong batas token.
   - **Tidak ada library eksternal:** cek dengan `grep -c "<script src=\|<link .*href=\"http" RUN_DIR/contoh-*.html`, hasilnya seharusnya 0.
   - **Bahasa:** isi halaman memakai bahasa Indonesia.

   Laporkan hasilnya:
   - loss akhir dan lama training;
   - lokasi adapter;
   - penilaian singkat kedua website;
   - saran langkah berikutnya: bandingkan dengan `--model qwen2.5-coder-7b`, coba `--dataset mix` jika ada gejala lupa kemampuan umum, atau ekspor GGUF dengan perintah `gguf`.

## Menangani masalah

| Gejala | Tindakan |
|---|---|
| `stage: error` dengan `CUDA out of memory` | Di T4, script otomatis memakai `--load-in-4bit --max-seq-len 6144`. Kombinasi ini sudah terbukti jalan. Semua sampel dataset dijaga di bawah 6144 token, jadi jumlah yang dibuang seharusnya 0; laporkan jika tidak. Jika error terjadi dengan pengaturan lain, ulangi dengan `--load-in-4bit --max-seq-len 6144` dan beri tahu pengguna bahwa akurasi Qwen3.5 sedikit turun dengan 4-bit. Hasilnya disimpan di folder `...-4bit`. Untuk batas di bawah 6144, tanya pengguna dulu, karena sebagian besar sampel web akan terbuang. |
| Loss `nan`, atau `grad_norm` `nan` terus-menerus | Hentikan proses. Jalankan `pip install --upgrade unsloth unsloth_zoo`, lalu ulangi. Jika tetap `nan`, laporkan ke pengguna, jangan diakali dengan mengganti presisi sendiri. |
| `ModuleNotFoundError`, atau error versi transformers/trl | Jalankan `pip install --upgrade unsloth`. Qwen3.5 butuh transformers v5. |
| Gagal mengunduh model atau data (HTTP 429/5xx, timeout) | Tunggu sebentar lalu ulangi. Jika berulang, minta pengguna mengisi secret `HF_TOKEN` di Colab, atau meng-upload `train.jsonl` ke `/content`. |
| `Google Drive belum ter-mount` | Minta pengguna menjalankan sel *Mount Google Drive*. |
| `status.json` tidak berubah lebih dari 15 menit dan prosesnya hilang (`pgrep -f train_unsloth` kosong) | Lihat akhir `/content/train.log`. Jika tidak ada error, kemungkinan proses dimatikan. Jalankan ulang perintah `train`, dan training dilanjutkan dari checkpoint. |
| Pengguna meminta berhenti ("stop training") | Jalankan `pkill -f "train_unsloth.py train"`. Checkpoint terakhir tetap ada di Drive dan bisa dilanjutkan dengan perintah `train` yang sama. |
| Runtime Colab di-reset (`/content/train_unsloth.py` hilang) | Kamu juga ikut hilang, jadi pengguna yang menjalankan ulang notebook. Checkpoint di Drive tetap aman dan perintah `train` akan melanjutkannya. |
| `stage: error` jenis lain | Baca `hint` dan `traceback`. Perbaiki hanya jika penyebabnya jelas dan aman. Jika tidak, laporkan ke pengguna beserta ringkasan error-nya. |

## Batasan

- **Boleh tanpa bertanya:**
  - menjalankan `info`, `prepare`, `train`, dan `test`;
  - memantau proses;
  - melanjutkan dari checkpoint;
  - beralih ke `--load-in-4bit` atau menurunkan `--max-seq-len` sampai 6144 saat kehabisan memori;
  - memasang ulang atau meng-upgrade paket Python;
  - menghentikan training jika pengguna memintanya.
- **Tanya dulu sebelum:**
  - mengganti model, dataset, atau hyperparameter lain;
  - beralih ke `--dataset mix`, karena sebagian sumbernya dibuat dengan model OpenAI dan ada implikasi lisensi (lihat README repo);
  - menghapus checkpoint atau hasil di Drive;
  - menjalankan `gguf`, karena butuh waktu sekitar 15–40 menit dan beberapa GB di Drive;
  - mengubah `/content/train_unsloth.py` atau file data.
- **Jangan pernah:**
  - menampilkan, menyalin, atau menyimpan isi secret (`HF_TOKEN` dan sejenisnya) ke file atau log;
  - menjalankan SSH, tunnel (ngrok dan sejenisnya), remote desktop, atau web UI. Kebijakan Colab membatasi hal-hal ini, terutama di tier gratis.

### B2. Pasang Claude Code dan buka terminal

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash
!pip install -q colab-xterm
%load_ext colabxterm
%xterm

### B3. Di terminal

1. Ketik perintah ini lalu Enter:
   ```
   cd /content && export PATH="$HOME/.local/bin:$PATH" && claude
   ```
2. **Login:**
   - Pilih login dengan akun Claude. Buka link yang muncul di tab baru, login, lalu salin kode yang diberikan.
   - Tempel kode itu di terminal (Ctrl+Shift+V atau klik kanan → Paste).
   - Jika muncul pertanyaan *trust this folder*, pilih **Yes**.
3. Ketik perintah untuk Claude:
   ```
   Latih model sesuai CONFIG di train_unsloth.py mengikuti CLAUDE.md, pantau sampai selesai, lalu laporkan hasilnya.
   ```
4. **Izin:** saat Claude meminta izin menjalankan perintah, pilih **Yes, and don't ask again** untuk perintah yang aman
   seperti `python /content/train_unsloth.py`, `tail`, `cat`, `nvidia-smi`, dan `sleep`.
5. Biarkan tab Colab tetap terbuka. Anda bisa bertanya kapan saja, misalnya `gimana progresnya?`, atau meminta
   `stop training`.

Training berjalan di latar belakang, jadi tetap jalan walaupun Anda keluar dari Claude atau terminal tertutup. Untuk
kembali memantau, jalankan ulang sel B2, lalu `cd /content && claude --continue`.

---
## Opsional: ekspor GGUF untuk Ollama / llama.cpp
Jalankan setelah training selesai. Butuh ±15–40 menit dan ±3 GB ruang di Drive (untuk 4B).

In [ ]:
!python /content/train_unsloth.py gguf

---
## Chat dengan model (Gradio)
Jalankan setelah training selesai, dan pastikan tidak ada training yang sedang berjalan karena GPU-nya dipakai bersama.
Adapter terbaru di `finetune-id/` dipilih otomatis. Ganti `CONFIG["adapter"]` di sel kedua untuk memilih adapter
lain, atau `"none"` untuk model dasar. Tampilan chat muncul di bawah sel ketiga.

In [ ]:
%%capture
!pip install "gradio>=6.30,<7"

In [ ]:
%%writefile /content/chat_gradio.py
"""Chatbot Gradio untuk model hasil fine-tuning (Qwen3.5-4B + adapter LoRA), dengan pengaturan lengkap.

Fitur: streaming, mode berpikir, preset instruksi sistem, sampling lengkap, perbandingan adapter vs model dasar tanpa
memuat ulang, ulangi/batalkan/edit pesan, tombol stop, pratinjau website dari blok ```html, unduh HTML dan percakapan,
serta statistik token per detik.

Cara memakai di Google Colab (GPU T4 cukup):

    Sel 1:  !pip install --upgrade unsloth "gradio>=6.30,<7"
    Sel 2:  from google.colab import drive; drive.mount("/content/drive")
            !wget -q -O /content/chat_gradio.py https://raw.githubusercontent.com/PvzKing/Dataset/main/scripts/chat_gradio.py
    Sel 3:  %run /content/chat_gradio.py

Tampilan chat muncul di bawah sel 3. Opsi CONFIG bisa diganti lewat argumen, misalnya:

    %run /content/chat_gradio.py --adapter /content/drive/MyDrive/finetune-id/qwen3.5-4b-own-4bit-lanjut/lora-adapter
    %run /content/chat_gradio.py --adapter none        # model dasar saja
    %run /content/chat_gradio.py --demo                # coba tampilan tanpa GPU dan tanpa model

Di luar Colab: python chat_gradio.py, lalu buka http://localhost:7860.
"""
import argparse
import builtins
import contextlib
import html
import json
import re
import sys
import tempfile
import threading
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

# ======================================================== CONFIG: ubah di sini ========================================
CONFIG = {
    "adapter": "auto",              # "auto" = adapter terbaru di folder out, path folder lora-adapter, atau "none"
    "out": "/content/drive/MyDrive/finetune-id",  # folder hasil training (dipakai saat adapter = "auto")
    "base_model": "unsloth/Qwen3.5-4B",           # dipakai jika adapter = "none"
    "load_in_4bit": None,           # None = otomatis: 4-bit di GPU tanpa bf16 asli (T4), selain itu 16-bit
    "max_seq_len": 8192,            # panjang konteks maksimum (riwayat + jawaban)
    "app_name": "Asisten Nusantara",
    "timezone": "Asia/Jakarta",     # untuk sapaan pagi/siang/sore/malam
    "share": False,                 # True = link publik *.gradio.live (wajib pakai auth)
    "auth": None,                   # "user:sandi"; wajib jika share = True
    "port": None,                   # None = port kosong pertama mulai 7860
    "demo": False,                  # True = jawaban tiruan, untuk mencoba tampilan tanpa GPU
}
# ======================================================================================================================

SYSTEM_PRESETS = {
    "Tanpa instruksi (seperti data training)": "",
    "Asisten umum": "Kamu asisten berbahasa Indonesia yang membantu, jujur, dan ringkas. Jika tidak yakin, katakan "
                    "tidak yakin daripada mengarang.",
    "Web developer (satu file HTML)": "Kamu web developer senior. Buat website lengkap dalam satu file HTML dengan CSS "
                                      "dan JavaScript inline, responsif, aksesibel, dan tanpa library eksternal. Setelah "
                                      "kode, tulis catatan singkat dan bagian **Cek kebutuhan** yang memetakan setiap "
                                      "permintaan ke bagian kode yang mengerjakannya.",
    "Analisis hukum Indonesia": "Kamu membantu analisis hukum Indonesia. Pisahkan fakta, isu hukum, dasar hukum, "
                                "analisis, dan kesimpulan. Sebut peraturan beserta pasal dan ayat hanya jika kamu yakin; "
                                "jika tidak yakin nomornya, katakan demikian. Ingatkan bahwa jawaban ini bukan nasihat "
                                "hukum dan perlu diperiksa ke sumber resmi.",
    "Kustom": None,
}

# Rekomendasi sampling Qwen3/Qwen3.5: (temperature, top_p, top_k, min_p)
QWEN_SAMPLING = {False: (0.7, 0.8, 20, 0.0), True: (0.6, 0.95, 20, 0.0)}

EXAMPLES = [
    "Buatkan landing page satu file HTML untuk bengkel motor \"Gaspol Motor\" di Yogyakarta: hero dengan tombol booking, "
    "daftar layanan dan harga, jam buka, dan form booking dengan validasi. Responsif, tanpa library eksternal.",
    "Buat aplikasi absensi kelas dalam satu file HTML: tambah siswa, tandai hadir/izin/sakit/alfa per tanggal, rekap "
    "persentase, simpan di localStorage, dan ekspor CSV.",
    "Jelaskan perbedaan wanprestasi dan perbuatan melawan hukum, lengkap dengan dasar hukumnya.",
    "Jelaskan konsep big-O dengan analogi sehari-hari.",
]

IN_NOTEBOOK = "ipykernel" in sys.modules
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="chat_gradio_"))


# ---------------------------------------------------------------- argumen

def parse_args():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--adapter", help='folder lora-adapter, "auto", atau "none"')
    parser.add_argument("--out", help="folder hasil training untuk adapter auto")
    parser.add_argument("--base-model")
    parser.add_argument("--load-in-4bit", action=argparse.BooleanOptionalAction, default=None)
    parser.add_argument("--max-seq-len", type=int)
    parser.add_argument("--app-name")
    parser.add_argument("--timezone")
    parser.add_argument("--share", action="store_true", default=None, help="buat link publik (wajib --auth)")
    parser.add_argument("--auth", help="user:sandi")
    parser.add_argument("--port", type=int)
    parser.add_argument("--demo", action="store_true", default=None, help="jawaban tiruan tanpa GPU")
    # Saat ditempel ke sel notebook, sys.argv berisi argumen kernel Jupyter: abaikan yang tidak dikenal.
    args, _ = parser.parse_known_args(sys.argv[1:])
    for key, value in CONFIG.items():
        if getattr(args, key, None) is None:
            setattr(args, key, value)
    if args.share and not args.auth:
        raise SystemExit("--share membuat chat bisa diakses siapa pun yang tahu link-nya: tambahkan --auth user:sandi")
    return args


# ---------------------------------------------------------------- model

def find_adapter(args):
    """Kembalikan folder adapter yang dipakai, atau None untuk model dasar."""
    if args.adapter == "none":
        return None
    if args.adapter != "auto":
        folder = Path(args.adapter)
        if not (folder / "adapter_config.json").is_file():
            raise SystemExit(f"bukan folder adapter LoRA (tidak ada adapter_config.json): {folder}")
        return folder
    found = sorted(Path(args.out).glob("*/lora-adapter/adapter_model.safetensors"), key=lambda p: p.stat().st_mtime)
    if not found:
        print(f"tidak ada adapter di {args.out}, memakai model dasar {args.base_model}", flush=True)
        return None
    return found[-1].parent


def native_bf16(torch):
    """bf16 asli (Ampere ke atas). torch.cuda.is_bf16_supported() bernilai True juga di T4 karena emulasi."""
    return torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8


class Engine:
    """Model, tokenizer, dan generasi streaming. Satu generasi berjalan pada satu waktu."""

    def __init__(self, args):
        self.args = args
        self.adapter = None
        self.lock = threading.Lock()
        self.current_stop = threading.Event()
        if args.demo:
            self.info = "**Mode demo**: jawaban tiruan, tanpa model."
            return
        import torch
        from unsloth import FastLanguageModel

        if args.load_in_4bit is None:
            args.load_in_4bit = not native_bf16(torch)
        self.adapter = find_adapter(args)
        name = str(self.adapter or args.base_model)
        print(f"memuat {name} (4-bit: {args.load_in_4bit}) ...", flush=True)
        load = dict(max_seq_length=args.max_seq_len, dtype=None, load_in_4bit=args.load_in_4bit,
                    load_in_16bit=not args.load_in_4bit)
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(model_name=name, **load)
        except Exception as e:
            if not self.adapter:
                raise
            # Cadangan: muat model dasar lalu pasang adapter dengan PEFT.
            base = json.loads((self.adapter / "adapter_config.json").read_text()).get("base_model_name_or_path")
            print(f"memuat adapter lewat Unsloth gagal ({e}); mencoba {base} + PEFT", flush=True)
            from peft import PeftModel
            model, tokenizer = FastLanguageModel.from_pretrained(model_name=base or args.base_model, **load)
            model = PeftModel.from_pretrained(model, str(self.adapter))
        FastLanguageModel.for_inference(model)
        self.model = model
        self.tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # processor multimodal -> tokenizer teks
        self.torch = torch
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        self.info = (f"**Model:** `{args.base_model if not self.adapter else self.adapter}`  \n"
                     f"**GPU:** {gpu} · {'4-bit' if args.load_in_4bit else '16-bit'} · konteks {args.max_seq_len:,} token")

    def stop(self):
        """Hentikan generasi yang sedang berjalan (tombol stop atau percakapan baru)."""
        self.current_stop.set()

    @property
    def has_adapter(self):
        return self.adapter is not None

    def count_tokens(self, text):
        if self.args.demo:
            return len(text) // 4
        return len(self.tokenizer(text, add_special_tokens=False)["input_ids"])

    def render(self, messages, thinking):
        if self.args.demo:
            return "\n".join(m["content"] for m in messages)
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                  enable_thinking=thinking)

    def stream(self, prompt, settings, use_adapter, stop_event):
        """Hasilkan potongan teks jawaban satu per satu."""
        if self.args.demo:
            yield from demo_stream(prompt, settings, stop_event)
            return
        from transformers import StoppingCriteria, StoppingCriteriaList, TextIteratorStreamer

        class StopOnEvent(StoppingCriteria):
            def __call__(self, input_ids, scores, **kwargs):
                return stop_event.is_set()

        torch = self.torch
        inputs = self.tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(self.model.device)
        # Token khusus dibiarkan supaya </think> tetap terlihat, lalu token <|...|> dibuang di split_thinking.
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=False)
        sample = settings["temperature"] > 0
        kwargs = dict(**inputs, streamer=streamer, max_new_tokens=settings["max_new_tokens"], do_sample=sample,
                      repetition_penalty=settings["repetition_penalty"],
                      stopping_criteria=StoppingCriteriaList([StopOnEvent()]))
        if sample:
            kwargs.update(temperature=settings["temperature"], top_p=settings["top_p"], top_k=settings["top_k"],
                          min_p=settings["min_p"])
        if settings["seed"] >= 0:
            torch.manual_seed(settings["seed"])
        can_disable = self.has_adapter and hasattr(self.model, "disable_adapter")
        adapter_off = self.model.disable_adapter() if can_disable and not use_adapter else contextlib.nullcontext()
        error = []

        def run():
            try:
                with adapter_off, torch.inference_mode():
                    self.model.generate(**kwargs)
            except Exception as e:  # tampilkan ke pengguna, jangan biarkan streamer menunggu selamanya
                error.append(e)
                streamer.end()

        with self.lock:
            thread = threading.Thread(target=run, daemon=True)
            thread.start()
            try:
                yield from streamer
            finally:
                stop_event.set()  # dibatalkan (tombol stop/tutup tab): hentikan generate di GPU
                thread.join()
        if error:
            raise error[0]


def demo_stream(prompt, settings, stop_event):
    thinking = "Pengguna meminta contoh. Aku jawab singkat dengan satu blok HTML supaya pratinjau bisa dicoba.</think>"
    answer = ("Berikut contoh halaman kecil untuk mencoba **pratinjau website**.\n\n```html\n<!DOCTYPE html>\n"
              "<html lang=\"id\"><head><meta charset=\"UTF-8\"><title>Demo</title><style>body{font-family:system-ui;"
              "display:grid;place-items:center;height:100vh;margin:0;background:#f5f4ed}button{padding:.8rem 1.4rem;"
              "border:0;border-radius:10px;background:#c96442;color:#fff;font-size:1rem}</style></head><body>"
              "<button id=\"b\">Klik saya</button><script>let n=Number(localStorage.getItem('n')||0);"
              "const b=document.getElementById('b');b.onclick=()=>{n++;localStorage.setItem('n',n);"
              "b.textContent=`Diklik ${n} kali`};</script></body></html>\n```\n\n"
              "Catatan:\n- Tombol → menghitung klik dan menyimpannya di `localStorage`.\n\n"
              "| Pengaturan | Nilai |\n|---|---|\n"
              f"| temperature | {settings['temperature']} |\n| max token | {settings['max_new_tokens']} |\n")
    text = (thinking if settings["thinking"] else "") + answer
    for i in range(0, len(text), 6):
        if stop_event.is_set():
            return
        time.sleep(0.01)
        yield text[i:i + 6]


# ---------------------------------------------------------------- percakapan

def text_of(content):
    """Isi pesan Gradio bisa berupa string atau daftar potongan {'type': 'text', 'text': ...}."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    return "".join(part.get("text", "") for part in content or [] if isinstance(part, dict))


def is_thought(message):
    return bool((message.get("metadata") or {}).get("title"))


def to_model_messages(history, system):
    """Riwayat Gradio -> pesan untuk model: tanpa proses berpikir, pesan berurutan dari role yang sama digabung."""
    messages = [{"role": "system", "content": system}] if system.strip() else []
    for m in history:
        if is_thought(m):
            continue
        text = text_of(m["content"])
        if messages and messages[-1]["role"] == m["role"]:
            messages[-1]["content"] += "\n\n" + text
        else:
            messages.append({"role": m["role"], "content": text})
    return messages


def fit_context(engine, messages, thinking, max_new_tokens):
    """Buang pasangan tanya-jawab terlama sampai prompt + jawaban muat di konteks. Kembalikan (prompt, n_token, dibuang)."""
    limit = engine.args.max_seq_len - max_new_tokens
    dropped = 0
    while True:
        prompt = engine.render(messages, thinking)
        n = engine.count_tokens(prompt)
        first = 1 if messages and messages[0]["role"] == "system" else 0
        if n <= limit or len(messages) - first <= 1:
            return prompt, n, dropped
        del messages[first:first + 2]
        dropped += 1


SPECIAL_TOKEN = re.compile(r"<\|[a-z_]+\|>")


def split_thinking(raw, thinking):
    """Pisahkan proses berpikir dan jawaban. Kembalikan (pikiran, jawaban, pikiran_selesai)."""
    raw = SPECIAL_TOKEN.sub("", raw)
    if not thinking:
        return "", raw, True
    raw = raw.removeprefix("<think>").lstrip("\n")
    if "</think>" in raw:
        thought, answer = raw.split("</think>", 1)
        return thought.strip(), answer.lstrip("\n"), True
    return raw, "", False


def last_html(history):
    for m in reversed(history):
        if m["role"] == "assistant" and not is_thought(m):
            blocks = re.findall(r"```html\s*\n(.*?)(?:```|$)", text_of(m["content"]), re.S)
            if blocks:
                return blocks[-1].strip()
    return None


# localStorage tidak tersedia di iframe sandbox tanpa allow-same-origin; ganti dengan penyimpanan di memori supaya
# aplikasi buatan model tetap jalan, tanpa memberi iframe akses ke halaman chat.
STORAGE_SHIM = ("<script>(()=>{try{window.localStorage.length}catch(e){const mk=()=>{const m=new Map();return{"
                "getItem:k=>m.has(String(k))?m.get(String(k)):null,setItem:(k,v)=>m.set(String(k),String(v)),"
                "removeItem:k=>m.delete(String(k)),clear:()=>m.clear(),key:i=>[...m.keys()][i]??null,"
                "get length(){return m.size}}};for(const n of['localStorage','sessionStorage'])"
                "Object.defineProperty(window,n,{value:mk(),configurable:true})}})()</script>")


def preview_html(page):
    if not page:
        return '<div class="preview-empty">Pratinjau muncul di sini saat jawaban berisi kode HTML.</div>'
    doc = STORAGE_SHIM + page
    return (f'<iframe class="preview-frame" title="Pratinjau website" '
            f'sandbox="allow-scripts allow-forms allow-modals allow-downloads allow-popups" '
            f'srcdoc="{html.escape(doc, quote=True)}"></iframe>')


def export_files(history):
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    md, data = [], []
    for m in history:
        text = text_of(m["content"])
        if is_thought(m):
            md.append(f"<details><summary>Proses berpikir</summary>\n\n{text}\n\n</details>\n")
            continue
        md.append(f"### {'Anda' if m['role'] == 'user' else 'Asisten'}\n\n{text}\n")
        data.append({"role": m["role"], "content": text})
    md_path = EXPORT_DIR / f"percakapan-{stamp}.md"
    md_path.write_text("\n".join(md), encoding="utf-8")
    json_path = EXPORT_DIR / f"percakapan-{stamp}.json"
    json_path.write_text(json.dumps({"messages": data}, ensure_ascii=False, indent=1), encoding="utf-8")
    return [str(md_path), str(json_path)]


def greeting(args):
    hour = datetime.now(ZoneInfo(args.timezone)).hour
    part = "pagi" if 4 <= hour < 11 else "siang" if hour < 15 else "sore" if hour < 18 else "malam"
    return f"Selamat {part}"


# ---------------------------------------------------------------- tampilan

THEME_FONTS = ("Inter", "Source Serif 4")

CSS = """
:root, .light {
  --cc-bg: #f5f4ed; --cc-sidebar: #efede4; --cc-surface: #ffffff; --cc-user: #e9e6dc; --cc-text: #2b2a26;
  --cc-muted: #6f6c62; --cc-border: #dedad0; --cc-accent: #c96442; --cc-accent-hover: #b4553a; --cc-code: #f0eee6;
}
.dark {
  --cc-bg: #262624; --cc-sidebar: #1f1e1d; --cc-surface: #30302e; --cc-user: #141413; --cc-text: #ecebe6;
  --cc-muted: #a6a39a; --cc-border: #3e3d39; --cc-accent: #d97757; --cc-accent-hover: #e08a6e; --cc-code: #1f1e1d;
}
body, gradio-app, .gradio-container { background: var(--cc-bg) !important; color: var(--cc-text); }
.gradio-container { max-width: 100% !important; padding: 0 !important; }
footer { display: none !important; }

/* sidebar */
.sidebar { background: var(--cc-sidebar) !important; border-right: 1px solid var(--cc-border) !important; }
.brand { display: flex; align-items: center; gap: .6rem; padding: .25rem 0 .75rem; }
.brand-mark { width: 30px; height: 30px; border-radius: 9px; background: var(--cc-accent); color: #fff;
  display: grid; place-items: center; font-family: "Source Serif 4", Georgia, serif; font-weight: 600; font-size: 17px; }
.brand-name { font-family: "Source Serif 4", Georgia, serif; font-size: 1.15rem; font-weight: 600; color: var(--cc-text); }

/* kolom chat */
#chat-col { max-width: 820px; margin: 0 auto; width: 100%; padding: 0 16px; }
#chatbot { background: transparent !important; border: 0 !important; box-shadow: none !important; }
#chatbot .message-row { background: transparent !important; }
#chatbot .message.user, #chatbot .message.user .message-content {
  background: var(--cc-user) !important; border: 0 !important; border-radius: 16px !important; color: var(--cc-text); }
#chatbot .bot-row .message, #chatbot .message.bot { background: transparent !important; border: 0 !important;
  box-shadow: none !important; max-width: 100% !important; width: 100%; padding-left: 0 !important; }
#chatbot .bot-row .flex-wrap, #chatbot .bot-row .role { max-width: 100% !important; width: 100%; }
#chatbot .message.bot .message-content, #chatbot .message.bot .prose, #chatbot .message.bot .md {
  font-family: "Source Serif 4", Georgia, serif !important; font-size: 1.06rem; line-height: 1.72; color: var(--cc-text); }
#chatbot .message.bot .prose h1, #chatbot .message.bot .prose h2, #chatbot .message.bot .prose h3 {
  font-family: "Source Serif 4", Georgia, serif !important; color: var(--cc-text); }
#chatbot .icon-button-wrapper, #chatbot .icon-button { background: transparent !important; border-color: transparent !important; }
#chatbot .message.bot table { font-family: Inter, system-ui, sans-serif; font-size: .92rem; }
#chatbot pre, #chatbot code { background: var(--cc-code) !important; border-radius: 8px; }
#chatbot .placeholder-content, #chatbot .placeholder { color: var(--cc-text); }
.hello { text-align: center; padding-top: 12vh; }
.hello .mark { width: 44px; height: 44px; border-radius: 12px; background: var(--cc-accent); color: #fff;
  display: inline-grid; place-items: center; font-family: "Source Serif 4", Georgia, serif; font-size: 24px; }
.hello h1 { font-family: "Source Serif 4", Georgia, serif; font-weight: 400; font-size: 2.3rem; margin: .8rem 0 .3rem;
  color: var(--cc-text); }
.hello p { color: var(--cc-muted); margin: 0; }

/* kotak ketik */
#composer { background: var(--cc-surface) !important; border: 1px solid var(--cc-border) !important;
  border-radius: 18px !important; box-shadow: 0 4px 18px rgba(0,0,0,.06) !important; padding: 6px 8px !important; }
#composer textarea { background: transparent !important; border: 0 !important; box-shadow: none !important;
  font-size: 1rem; color: var(--cc-text); }
#composer button.submit-button, #composer .submit-button { background: var(--cc-accent) !important;
  border-radius: 10px !important; color: #fff !important; }
#composer button.submit-button:hover { background: var(--cc-accent-hover) !important; }
.stats, .stats * { font-size: .78rem !important; color: var(--cc-muted) !important; text-align: center; }
.disclaimer, .disclaimer * { font-size: .74rem !important; color: var(--cc-muted) !important; text-align: center; }
.model-info, .model-info * { font-size: .8rem !important; color: var(--cc-muted) !important; }

/* pratinjau */
#preview-col { border-left: 1px solid var(--cc-border); padding: 0 12px !important; }
.preview-frame { width: 100%; height: 72vh; border: 1px solid var(--cc-border); border-radius: 12px; background: #fff; }
.preview-empty { height: 72vh; display: grid; place-items: center; text-align: center; color: var(--cc-muted);
  border: 1px dashed var(--cc-border); border-radius: 12px; padding: 1rem; }

/* tombol */
.cc-btn { border-radius: 10px !important; }
.cc-primary { background: var(--cc-accent) !important; color: #fff !important; border: 0 !important; }
.cc-primary:hover { background: var(--cc-accent-hover) !important; }
"""

TOGGLE_DARK_JS = "() => { document.body.classList.toggle('dark'); }"
# Di layar sempit sidebar menutupi chat: tutup otomatis saat halaman dibuka.
CLOSE_SIDEBAR_ON_PHONE_JS = """() => { if (window.innerWidth < 768) {
  const b = document.querySelector('.sidebar.open .toggle-button'); if (b) b.click(); } }"""


def build_theme(gr):
    accent = gr.themes.Color(c50="#fbf1ec", c100="#f6dfd4", c200="#eebfaa", c300="#e39d80", c400="#d97757",
                             c500="#c96442", c600="#b4553a", c700="#944530", c800="#743728", c900="#552920",
                             c950="#3a1c16", name="terracotta")
    stone = gr.themes.Color(c50="#faf9f5", c100="#f5f4ed", c200="#e9e6dc", c300="#dedad0", c400="#b7b3a7",
                            c500="#8f8b80", c600="#6f6c62", c700="#3e3d39", c800="#30302e", c900="#262624",
                            c950="#1f1e1d", name="stone_warm")
    theme = gr.themes.Base(
        primary_hue=accent, secondary_hue=accent, neutral_hue=stone, radius_size=gr.themes.sizes.radius_lg,
        font=[gr.themes.GoogleFont(THEME_FONTS[0]), "system-ui", "sans-serif"],
        font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
    )
    return theme.set(
        body_background_fill="#f5f4ed", body_background_fill_dark="#262624",
        body_text_color="#2b2a26", body_text_color_dark="#ecebe6",
        background_fill_primary="#ffffff", background_fill_primary_dark="#30302e",
        background_fill_secondary="#efede4", background_fill_secondary_dark="#1f1e1d",
        block_background_fill="#faf9f5", block_background_fill_dark="#2b2b29",
        block_border_color="#dedad0", block_border_color_dark="#3e3d39",
        input_background_fill="#ffffff", input_background_fill_dark="#30302e",
        button_primary_background_fill="#c96442", button_primary_background_fill_hover="#b4553a",
        button_primary_background_fill_dark="#d97757", button_primary_text_color="#ffffff",
        slider_color="#c96442", slider_color_dark="#d97757",
        checkbox_background_color_selected="#c96442", checkbox_background_color_selected_dark="#d97757",
    )


def build_app(engine, args):
    import gradio as gr

    initial = args.app_name[:1].upper()
    hello = (f'<div class="hello"><div class="mark">{html.escape(initial)}</div>'
             f'<h1>{greeting(args)}</h1><p>Ada yang bisa dibantu hari ini?</p></div>')

    with gr.Blocks(title=args.app_name, fill_height=True) as demo:
        with gr.Sidebar(open=not IN_NOTEBOOK, width=330, elem_classes="sidebar"):
            gr.HTML(f'<div class="brand"><div class="brand-mark">{html.escape(initial)}</div>'
                    f'<div class="brand-name">{html.escape(args.app_name)}</div></div>')
            new_chat = gr.Button("＋  Percakapan baru", elem_classes=["cc-btn", "cc-primary"])
            with gr.Accordion("Model", open=True):
                use_adapter = gr.Radio(["Fine-tune (adapter)", "Model dasar"], value="Fine-tune (adapter)",
                                       label="Bobot yang dipakai", interactive=engine.has_adapter or args.demo,
                                       info="Bandingkan hasil fine-tune dengan model dasar tanpa memuat ulang.")
                thinking = gr.Checkbox(False, label="Mode berpikir (thinking)",
                                       info="Model dilatih tanpa mode berpikir; aktifkan untuk eksperimen.")
                gr.Markdown(engine.info, elem_classes="model-info")
            with gr.Accordion("Instruksi sistem", open=False):
                preset = gr.Dropdown(list(SYSTEM_PRESETS), value=next(iter(SYSTEM_PRESETS)), label="Preset")
                system = gr.Textbox("", lines=5, label="Isi instruksi", placeholder="Kosong = tanpa instruksi sistem")
            with gr.Accordion("Sampling", open=False):
                temperature = gr.Slider(0, 2, 0.7, step=0.05, label="Temperature", info="0 = deterministik (greedy)")
                top_p = gr.Slider(0.05, 1, 0.8, step=0.01, label="Top-p")
                top_k = gr.Slider(0, 200, 20, step=1, label="Top-k", info="0 = nonaktif")
                min_p = gr.Slider(0, 0.5, 0.0, step=0.01, label="Min-p")
                repetition_penalty = gr.Slider(1.0, 1.5, 1.05, step=0.01, label="Repetition penalty")
                seed = gr.Number(-1, precision=0, label="Seed", info="-1 = acak")
                recommended = gr.Button("Pakai rekomendasi Qwen", size="sm", elem_classes="cc-btn")
            with gr.Accordion("Panjang & konteks", open=False):
                max_new_tokens = gr.Slider(64, 8192, 6000, step=64, label="Maks. token jawaban",
                                           info="Website satu file butuh ±4.000–6.000 token")
                gr.Markdown(f"Konteks total {args.max_seq_len:,} token. Riwayat terlama dibuang otomatis jika "
                            "tidak muat.", elem_classes="model-info")
            with gr.Accordion("Tampilan & ekspor", open=False):
                show_preview = gr.Checkbox(True, label="Panel pratinjau website")
                dark = gr.Button("🌓  Mode gelap/terang", size="sm", elem_classes="cc-btn")
                export = gr.Files(label="Unduh percakapan (.md / .json)", interactive=False)

        with gr.Row(equal_height=False):
            with gr.Column(scale=3, elem_id="chat-col"):
                chatbot = gr.Chatbot(
                    elem_id="chatbot", show_label=False, layout="bubble", placeholder=hello, editable="user",
                    buttons=["copy", "copy_all"], height="74vh", autoscroll=True,
                    examples=[{"text": e} for e in EXAMPLES], group_consecutive_messages=False,
                    latex_delimiters=[{"left": "$$", "right": "$$", "display": True}],
                )
                composer = gr.Textbox(
                    # lines=1 + max_lines>1: Enter mengirim, Shift+Enter baris baru, kotak memanjang otomatis.
                    elem_id="composer", show_label=False, lines=1, max_lines=12, autofocus=True, container=False,
                    placeholder=f"Tulis pesan untuk {args.app_name}…  (Enter kirim, Shift+Enter baris baru)",
                    submit_btn=True, stop_btn=True,
                )
                stats = gr.Markdown("", elem_classes="stats")
                gr.Markdown(f"{args.app_name} dapat keliru. Periksa kembali jawaban penting, terutama dasar hukum "
                            "dan kode.", elem_classes="disclaimer")
            with gr.Column(scale=2, elem_id="preview-col") as preview_col:
                gr.Markdown("#### Pratinjau website")
                preview = gr.HTML(preview_html(None))
                html_file = gr.DownloadButton("Unduh HTML", visible=False, elem_classes="cc-btn")

        settings_inputs = [system, thinking, temperature, top_p, top_k, min_p, repetition_penalty, seed,
                           max_new_tokens, use_adapter]

        # -------------------------------------------------- logika

        def respond(history, system, thinking, temperature, top_p, top_k, min_p, repetition_penalty, seed,
                    max_new_tokens, use_adapter):
            settings = dict(thinking=thinking, temperature=float(temperature), top_p=float(top_p), top_k=int(top_k),
                            min_p=float(min_p), repetition_penalty=float(repetition_penalty),
                            seed=int(seed if seed is not None else -1), max_new_tokens=int(max_new_tokens))
            messages = to_model_messages(history, system or "")
            if not messages or messages[-1]["role"] != "user":
                yield history, ""
                return
            prompt, n_prompt, dropped = fit_context(engine, messages, thinking, settings["max_new_tokens"])
            if dropped:
                gr.Warning(f"{dropped} pasangan tanya-jawab terlama tidak dikirim ke model karena konteks penuh.")
            stop_event = engine.current_stop = threading.Event()
            start, raw, first_token = time.time(), "", None
            thought_msg = {"role": "assistant", "content": "", "metadata": {"title": "💭 Proses berpikir",
                                                                             "status": "pending"}}
            answer_msg = {"role": "assistant", "content": ""}
            try:
                for piece in engine.stream(prompt, settings, use_adapter.startswith("Fine"), stop_event):
                    first_token = first_token or time.time()
                    raw += piece
                    thought, answer, done = split_thinking(raw, thinking)
                    shown = []
                    if thinking:
                        thought_msg["content"] = thought or "…"
                        if done:
                            thought_msg["metadata"]["status"] = "done"
                            thought_msg["metadata"]["duration"] = round(time.time() - start, 1)
                        shown.append(thought_msg)
                    if answer or not thinking:
                        answer_msg["content"] = answer
                        shown.append(answer_msg)
                    yield history + shown, "⏳ menulis…"
            finally:
                stop_event.set()
            elapsed = time.time() - start
            n_out = engine.count_tokens(SPECIAL_TOKEN.sub("", raw))
            speed = n_out / (time.time() - first_token) if first_token and time.time() > first_token else 0
            label = "dasar" if not use_adapter.startswith("Fine") else ("adapter" if engine.has_adapter else "dasar")
            info = (f"{n_out:,} token · {speed:.1f} token/detik · {elapsed:.0f} detik · prompt {n_prompt:,} token · "
                    f"model {label}")
            if n_out >= settings["max_new_tokens"] - 1:
                info += " · ⚠️ terpotong di batas maks. token"
            thought, answer, _ = split_thinking(raw, thinking)
            final = []
            if thinking and thought:
                thought_msg["content"] = thought
                thought_msg["metadata"]["status"] = "done"
                final.append(thought_msg)
            final.append({"role": "assistant", "content": answer or "_(jawaban kosong)_"})
            yield history + final, info

        def add_user(text, history):
            text = (text or "").strip()
            if not text:
                return gr.update(), history
            return "", history + [{"role": "user", "content": text}]

        def after_reply(history, show):
            page = last_html(history)
            files = export_files(history) if history else None
            if page:
                path = EXPORT_DIR / f"website-{datetime.now().strftime('%Y%m%d-%H%M%S')}.html"
                path.write_text(page, encoding="utf-8")
                return gr.update(value=preview_html(page)), gr.update(value=str(path), visible=True), files
            return gr.update(), gr.update(), files

        def trim_to_last_user(history):
            while history and history[-1]["role"] != "user":
                history = history[:-1]
            return history

        def on_retry(history):
            return trim_to_last_user(history)

        def on_undo(history):
            last = max((i for i, m in enumerate(history) if m["role"] == "user"), default=None)
            if last is None:
                return history, gr.update()
            return history[:last], text_of(history[last]["content"])

        def on_edit(history, evt: gr.EditData):
            history = history[:evt.index + 1]
            history[-1] = {"role": "user", "content": evt.value if isinstance(evt.value, str) else text_of(evt.value)}
            return history

        def on_example(history, evt: gr.SelectData):
            value = evt.value
            text = value.get("text", "") if isinstance(value, dict) else str(value)
            return history + [{"role": "user", "content": text}]

        def on_preset(name, current):
            value = SYSTEM_PRESETS.get(name)
            return current if value is None else value

        def on_recommended(thinking):
            t, p, k, m = QWEN_SAMPLING[bool(thinking)]
            return t, p, k, m, 1.0 if thinking else 1.05

        run = dict(fn=respond, inputs=[chatbot, *settings_inputs], outputs=[chatbot, stats],
                   concurrency_limit=1, concurrency_id="gpu")
        done = dict(fn=after_reply, inputs=[chatbot, show_preview], outputs=[preview, html_file, export])

        generations = []
        for trigger in (composer.submit(add_user, [composer, chatbot], [composer, chatbot], queue=False),
                        chatbot.retry(on_retry, chatbot, chatbot, queue=False),
                        chatbot.edit(on_edit, chatbot, chatbot, queue=False),
                        chatbot.example_select(on_example, chatbot, chatbot, queue=False)):
            gen = trigger.then(**run)
            gen.then(**done)
            generations.append(gen)
        composer.stop(engine.stop, None, None, cancels=generations, queue=False)
        chatbot.undo(on_undo, chatbot, [chatbot, composer], queue=False)

        def reset():
            engine.stop()
            return [], "", gr.update(value=preview_html(None)), gr.update(visible=False), None

        new_chat.click(reset, None, [chatbot, stats, preview, html_file, export], cancels=generations, queue=False)
        preset.change(on_preset, [preset, system], system, queue=False)
        recommended.click(on_recommended, thinking, [temperature, top_p, top_k, min_p, repetition_penalty],
                          queue=False)
        show_preview.change(lambda v: gr.update(visible=v), show_preview, preview_col, queue=False)
        dark.click(None, None, None, js=TOGGLE_DARK_JS)
        demo.load(None, None, None, js=CLOSE_SIDEBAR_ON_PHONE_JS)
    return demo


def get_engine(args):
    """Di notebook, sel ini bisa dijalankan ulang: pakai lagi model yang sudah dimuat supaya VRAM tidak penuh."""
    key = (args.adapter, args.out, args.base_model, args.load_in_4bit, args.max_seq_len, args.demo)
    cached = getattr(builtins, "_chat_gradio_engine", None)
    if cached and cached[0] == key:
        print("memakai model yang sudah dimuat", flush=True)
        return cached[1]
    if cached:  # pengaturan model berubah: lepas model lama dulu
        delattr(builtins, "_chat_gradio_engine")
        del cached
        import gc
        gc.collect()
        if "torch" in sys.modules:
            sys.modules["torch"].cuda.empty_cache()
    engine = Engine(args)
    builtins._chat_gradio_engine = (key, engine)
    return engine


def main():
    args = parse_args()
    import gradio as gr

    gr.close_all()  # tutup tampilan lama jika sel dijalankan ulang
    engine = get_engine(args)
    demo = build_app(engine, args)
    auth = tuple(args.auth.split(":", 1)) if args.auth else None
    demo.queue(default_concurrency_limit=1).launch(
        share=args.share, auth=auth, server_port=args.port, theme=build_theme(gr), css=CSS,
        height=900, show_error=True, debug=not IN_NOTEBOOK,
    )
    return demo


if __name__ == "__main__":
    main()

In [ ]:
%run /content/chat_gradio.py